# 06 - Structural growth against leaf function

The results generated here were used in manuscript section 3.8. Growth is
measured per branch and per month; gas
exchange per leaf at eleven bimonthly campaigns. To correlate them they must
meet at a common unit, so both are aggregated to **campaign x cultivation
system x sex** - 42 groups.

That aggregation is also what makes apparent quantum yield usable here: Phi is
a slope, so it exists once per group rather than once per leaf, and at group
level it sits alongside the other seven traits.

**Produces:** Figure 8.

In [ ]:
import sys
from pathlib import Path

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(ROOT / "src"))

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy.stats import pearsonr

from yerbamate import config as C
from yerbamate import io_physio, plotting as P, stats as S

P.use_paper_style(scale=1.5)
pd.set_option("display.width", 170)

physio = pd.read_csv(C.PHYSIO_CLEAN)
growth = pd.read_csv(C.UNIFIED)

## Aggregate both datasets to the shared unit

In [ ]:
growth_agg = (growth[growth.time_idx.isin(C.PERIOD_TO_TIDX.values())]
              .groupby(["time_idx", "environment", "sex"])[C.MORPHO_VARS]
              .mean().reset_index().rename(columns={"sex": "Sexo"}))

rows = []
for (tidx, env, sexo), sub in physio.groupby(["time_idx", "environment", "Sexo"]):
    rec = {"time_idx": tidx, "environment": env, "Sexo": sexo}
    for v in C.PHYSIO_VARS:
        rec[v] = sub[v].mean()
    rec["AQY"] = io_physio.apparent_quantum_yield(sub)
    rows.append(rec)
physio_agg = pd.DataFrame(rows)

merged = growth_agg.merge(physio_agg, on=["time_idx", "environment", "Sexo"], how="inner")
print(f"{len(merged)} groups (campaign x system x sex)")
print(merged.groupby(["environment", "Sexo"]).size().to_string())

## Figure 8 - correlation matrix

Structural growth at branch scale tracks water and CO2 exchange at leaf scale:
shoot elongation and leaf number increase correlate positively with gs and E,
metamer emission with E, leaf area increase with gs.

The negative leaf-number / LUE correlation is the interesting one. LUE is
highest where PPFD is lowest - in the agroforestry shade - which is also where
leaf production is lowest. The correlation is a consequence of that shared
dependence on light, not a trade-off between the two processes.

WUE, iWUE, deltaT and Phi are uncorrelated with every morphogenetic trait.

In [ ]:
PHYSIO_ALL = C.PHYSIO_VARS + ["AQY"]
R = np.full((len(C.MORPHO_VARS), len(PHYSIO_ALL)), np.nan)
Pv = np.full_like(R, np.nan)
rows = []
for i, gv in enumerate(C.MORPHO_VARS):
    for j, pv in enumerate(PHYSIO_ALL):
        sub = merged[[gv, pv]].dropna()
        if len(sub) < 5:
            continue
        r, p = pearsonr(sub[gv], sub[pv])
        R[i, j], Pv[i, j] = r, p
        rows.append({"Growth": C.MORPHO_LABELS[gv],
                     "Physiology": C.PHYSIO_LABELS.get(pv, pv),
                     "r": round(r, 3), "p": round(p, 4), "n": len(sub), "sig": S.stars(p)})

corr = pd.DataFrame(rows)
corr.to_csv(C.TAB_DIR / "Figure_8_growth_physiology_correlations.csv", index=False)
print(corr[corr.sig != "n.s."].to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(13.5, 8))
im = ax.imshow(R, cmap="RdBu_r", vmin=-1, vmax=1, aspect="auto")
ax.set_xticks(range(len(PHYSIO_ALL)))
ax.set_xticklabels([C.PHYSIO_LABELS.get(p, p) for p in PHYSIO_ALL], fontsize=24)
ax.set_yticks(range(len(C.MORPHO_VARS)))
ax.set_yticklabels([C.MORPHO_LABELS[g] for g in C.MORPHO_VARS], fontsize=21)
for i in range(R.shape[0]):
    for j in range(R.shape[1]):
        if np.isnan(R[i, j]):
            continue
        mark = ("***" if Pv[i, j] < 0.001 else "**" if Pv[i, j] < 0.01
                else "*" if Pv[i, j] < 0.05 else "")
        ax.text(j, i, f"{R[i, j]:+.2f}\n{mark}", ha="center", va="center", fontsize=18,
                fontweight="bold" if Pv[i, j] < 0.05 else "normal",
                color="white" if abs(R[i, j]) > 0.55 else "#222")
cb = fig.colorbar(im, ax=ax, fraction=0.045, pad=0.03)
cb.set_label("Pearson r", fontsize=22)
cb.ax.tick_params(labelsize=19)
ax.set_xticks(np.arange(-0.5, len(PHYSIO_ALL), 1), minor=True)
ax.set_yticks(np.arange(-0.5, len(C.MORPHO_VARS), 1), minor=True)
ax.grid(which="minor", color="white", lw=2)
ax.tick_params(which="minor", length=0)
fig.tight_layout()
P.save(fig, "Figure_8", dpi=600)

In [ ]:
c = corr.set_index(["Growth", "Physiology"])
assert c.loc[("Shoot elongation", "gₛ"), "sig"] == "**"
assert c.loc[("Shoot elongation", "E"), "sig"] == "**"
assert c.loc[("Metamer emission", "E"), "sig"] == "*"
assert c.loc[("Leaf number increase", "LUE"), "r"] < 0
assert (c.loc[("Shoot elongation", "WUE"), "sig"] == "n.s."
        and c.loc[("Shoot elongation", "Φ"), "sig"] == "n.s.")
assert corr.n.max() == 42
print("Validated the Figure 8 correlation results used in the manuscript.")